In [9]:
import os
import random
import librosa
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from tqdm import tqdm

# === CONFIGURATION ===
SR = 16000
FRAME_DURATION = 4  # seconds
N_MFCC = 39
N_CLUSTERS = 100
RANDOM_STATE = 42
OUTPUT_CSV = 'BoAWMFCC_features_modma.csv'

# === Global lists ===
all_frames_features_list = []
failed_files = []

# === STEP 1: Extract MFCCs from each 4s frame ===
def process_audio_in_frames(audio_path, frame_duration=4, sr=16000, n_mfcc=39):
    try:
        y, _ = librosa.load(audio_path, sr=sr)
        frame_len = sr * frame_duration
        num_frames = len(y) // frame_len
        frame_mfccs = []

        for i in range(num_frames):
            start = i * frame_len
            end = start + frame_len
            frame = y[start:end]
            if len(frame) == frame_len:
                mfcc_feat = librosa.feature.mfcc(y=frame, sr=sr, n_mfcc=n_mfcc).T  # shape (time, n_mfcc)
                frame_mfccs.append(mfcc_feat)
        return frame_mfccs
    except Exception as e:
        print(f"❌ Error processing {audio_path}: {e}")
        return []


# === STEP 2: Process directories and collect all MFCC frames with labels ===
def process_directory(directory, label, all_frames_list, frame_duration=4):
    total_files = sum(len(files) for _, _, files in os.walk(directory))

    with tqdm(total=total_files, desc=f"Processing {directory}") as pbar:
        for subdir, _, files in os.walk(directory):
            for file in files:
                if file.lower().endswith('.wav'):
                    file_path = os.path.join(subdir, file)
                    frames_features = process_audio_in_frames(file_path, frame_duration=frame_duration)

                    if not frames_features:
                        failed_files.append(file_path)
                        pbar.update(1)
                        continue

                    for frame in frames_features:
                        all_frames_list.append((frame, label))

                    pbar.update(1)

# === STEP 3: Create KMeans codebook ===
def create_codebook(mfcc_data, n_clusters=100):
    print("🔄 Fitting KMeans on all MFCC vectors...")
    all_vectors = np.vstack(mfcc_data)
    kmeans = KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE, n_init='auto')
    kmeans.fit(all_vectors)
    return kmeans

# === STEP 4: Convert MFCCs to BoAW vectors ===
def extract_boaw_features(mfcc_frame, kmeans_model):
    cluster_ids = kmeans_model.predict(mfcc_frame)
    hist, _ = np.histogram(cluster_ids, bins=np.arange(kmeans_model.n_clusters + 1))
    hist = hist / np.sum(hist)  # normalize
    return hist

# === STEP 5: Main execution ===
def main():
    # Process HC and MDD directories
    process_directory(r"F:\MODMA\audio_lanzhou_2015\HC", 0, all_frames_features_list, frame_duration=FRAME_DURATION)
    process_directory(r"F:\MODMA\audio_lanzhou_2015\MDD", 1, all_frames_features_list, frame_duration=FRAME_DURATION)

    print(f"\n✅ Total valid frames collected: {len(all_frames_features_list)}")
    print(f"❌ Total failed files: {len(failed_files)}")

    # Prepare data for KMeans
    all_mfcc_vectors = [vec for mfcc_seq, _ in all_frames_features_list for vec in mfcc_seq]

    if len(all_mfcc_vectors) == 0:
        print("🚫 No MFCC vectors available for clustering. Exiting.")
        return

    # Train KMeans codebook
    kmeans = create_codebook(all_mfcc_vectors, n_clusters=N_CLUSTERS)

    # Create BoAW dataset
    X, y = [], []
    for mfcc_frame, label in tqdm(all_frames_features_list, desc="Extracting BoAW features"):
        boaw_vector = extract_boaw_features(mfcc_frame, kmeans)
        X.append(boaw_vector)
        y.append(label)

    # Save to CSV
    df = pd.DataFrame(X)
    df['label'] = y
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n✅ BoAWMFCC features saved to: {OUTPUT_CSV}")

if __name__ == "__main__":
    main()


Processing F:\MODMA\audio_lanzhou_2015\MDD:  16%|█████▊                              | 108/667 [00:01<00:11, 49.74it/s]C:\Users\Administrator\AppData\Local\Temp\ipykernel_35328\1403578888.py:24: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(audio_path, sr=sr)
C:\Users\Administrator\.conda\envs\p311\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
Processing F:\MODMA\audio_lanzhou_2015\MDD:  17%|██████▏                             | 114/667 [00:02<00:13, 39.66it/s]

❌ Error processing F:\MODMA\audio_lanzhou_2015\MDD\02010004\24.wav: 
❌ Error processing F:\MODMA\audio_lanzhou_2015\MDD\02010004\25.wav: 
❌ Error processing F:\MODMA\audio_lanzhou_2015\MDD\02010004\26.wav: 
❌ Error processing F:\MODMA\audio_lanzhou_2015\MDD\02010004\27.wav: 


Processing F:\MODMA\audio_lanzhou_2015\MDD:  18%|██████▍                             | 119/667 [00:02<00:15, 34.52it/s]

❌ Error processing F:\MODMA\audio_lanzhou_2015\MDD\02010004\28.wav: 


Processing F:\MODMA\audio_lanzhou_2015\MDD: 100%|████████████████████████████████████| 667/667 [00:10<00:00, 62.87it/s]



✅ Total valid frames collected: 5728
❌ Total failed files: 64
🔄 Fitting KMeans on all MFCC vectors...


Extracting BoAW features: 100%|███████████████████████████████████████████████████| 5728/5728 [00:18<00:00, 313.11it/s]



✅ BoAWMFCC features saved to: BoAWMFCC_features_modma.csv
